<a href="https://colab.research.google.com/github/HunterTzou/DATA_201_SPRING_2026/blob/main/Project%201/%20Tzou_Hunter_DATA201_Project1_Week8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 1: Montgomery County HHS Food Inspection Analysis
**Name:** Hunter Tzou  
**Course:** DATA 201  
**Submission Date:**

---
## 1. Introduction

*Write 1–2 paragraphs here covering:*
- *Dataset origin*
- *Population of interest*
- *Data collection method and limitations*
- *Ethical or bias concerns*

This dataset was retrieved from the Montgomery County Open Data Portal and originates from the Health & Human Services Licensure and Regulatory Services Division. Data is collected by inspectors in the field during food safety inspections at establishments where food is prepared and sold, and is updated automatically by the county's database platform.

According to an inspector interviewed as part of this project, data is entered into the county's system in real time during each inspection. One important structural note is that the dataset is organized at the checklist-item level — each inspection event generates approximately 104 rows, one per compliance criterion, which has direct implications for how the data must be wrangled. I also identified several data quality concerns: (1) there are no raw quantitative measurements — metrics like hot holding temperature are recorded only as In Compliance or Not In Compliance rather than numeric values; (2) there appear to be a substantial number of duplicate records; and (3) inspection numbers do not appear to be uniquely tied to individual registration numbers. These issues complicate basic counts like unique inspections or active establishments. I am working with county officials to clarify these ambiguities and am confident the dataset is suitable for the analyses conducted here.

Regarding bias and ethical concerns, inspection frequency may not be uniform across zip codes or business types — higher-risk establishments may be inspected more often, which could skew geographic comparisons. Additionally, since compliance is recorded categorically rather than as continuous measurements, nuanced differences in food safety performance may not be fully captured.

<br>

**Citation (ACM style):**

> Montgomery County Department of Health and Human Services. 2024. HHS Food Inspection Data from July 2024 and Onward. Montgomery County Open Data Portal. https://data.montgomerycountymd.gov/Health-and-Human-Services/HHS-Food-Inspection-Data-from-July-2024-and-onward/dkrp-gr48 (Accessed March 28, 2026).

---
## 2. Dataset Structure

In [3]:
# Computation & Exploration
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Geolocation
import folium
import ast

# Query from databases and access folder systems
import requests
import os

In [4]:
# --- Imports and Data Loading ---
# Load the dataset from the Socrata API (paginated)

# To set your token in an environment variable: in your terminal run -> export SOCRATA_TOKEN="your_token_here"

APP_TOKEN = os.getenv("SOCRATA_TOKEN")  # Returns None if not set — that's fine!

BASE_URL = "https://data.montgomerycountymd.gov/resource/dkrp-gr48.json"
LIMIT = 50000
all_records = []
offset = 0

print("Fetching data...")
while True:
    params = {
        "$limit": LIMIT,
        "$offset": offset,
    }

    # Only add token to request if it's available
    if APP_TOKEN:
        params["$$app_token"] = APP_TOKEN

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    batch = response.json()

    if not batch:
        break

    all_records.extend(batch)
    offset += len(batch)
    print(f"  Fetched {offset:,} rows so far...")

    if len(batch) < LIMIT:
        break

# Build the DataFrame
df = pd.DataFrame(all_records)
print(f"\nRaw row count: {len(df):,}")

# Parse the location dictionaries

def parse_location(loc):
    """Parses a stringified GeoJSON Point dict -> (lat, lon) or (None, None)."""
    try:
        point = ast.literal_eval(str(loc))          # safely parse the stringified dict
        lon, lat = point['coordinates']              # GeoJSON is [lon, lat] order
        return round(lat, 5), round(lon, 5)
    except Exception:
        return None, None

Fetching data...
  Fetched 50,000 rows so far...
  Fetched 100,000 rows so far...
  Fetched 150,000 rows so far...
  Fetched 200,000 rows so far...
  Fetched 250,000 rows so far...
  Fetched 300,000 rows so far...
  Fetched 350,000 rows so far...
  Fetched 400,000 rows so far...
  Fetched 450,000 rows so far...
  Fetched 500,000 rows so far...
  Fetched 550,000 rows so far...
  Fetched 600,000 rows so far...
  Fetched 650,000 rows so far...
  Fetched 700,000 rows so far...
  Fetched 750,000 rows so far...
  Fetched 800,000 rows so far...
  Fetched 850,000 rows so far...
  Fetched 900,000 rows so far...
  Fetched 950,000 rows so far...
  Fetched 1,000,000 rows so far...
  Fetched 1,028,729 rows so far...

Raw row count: 1,028,729


In [5]:
# --- Raw Dataset Size ---
# Print number of rows and columns

rows_cols_tuple = df.shape
print(f"DataFrame shape (rows, columns): {rows_cols_tuple}")


DataFrame shape (rows, columns): (1028729, 30)


In [30]:
# DEDUPLICATE DATA

# There are a significant number of duplicates, which I have confirmed by my own count using
# Apply location parser and drop the original dict column

df[['latitude', 'longitude']] = df['location'].apply(
    lambda x: pd.Series(parse_location(x))
)

df = df.drop(columns=['location'])

print(f"Duplicate rows: {df.duplicated().sum():,}")

## Drop the Duplicates

df_clean = df.drop_duplicates().copy()
print(f"After dedup: {len(df_clean):,} rows")

print(df_clean.dtypes)

After dedup: 16,264 rows
registration_number                     object
inspection_number                       object
business_name                           object
owner                                   object
address                                 object
city                                    object
state                                   object
zip                                     object
business_type                           object
business_subtype                        object
inspection_type                         object
inspection_start_date                   object
status                                  object
food_from_approved_source               object
food_protected_from_contamination       object
workers_restricted                      object
proper_hand_washing                     object
cooling_time_and_temperature            object
cold_holding_temperature                object
hot_holding_temperature                 object
cooking_time_and_temperature       

In [20]:
df_clean.head()

,registration_number,inspection_number,business_name,owner,address,city,state,zip,business_type,business_subtype,...,reheating_time_and_temperature,hot_and_cold_running_water_provided,proper_sewage_disposal,toxic_substances_and_pesticides,rodents_and_insects,nutritional_labeling,trans_fat_ban,nosmoking_sign_posted,latitude,longitude
0,40895,26-2062,"TAQUERIA EL REY, LLC #4",TAQUERIA EL REY,9015 2ND ST. LANHAM,SEABROOK,MD,20706,Food Service Facility,Food License-Mobile Food-New-High,...,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,38.97379,-76.85813
2,46486,26-2038,KUNG FU TEA,CREST GROUP MD INC,19717 FREDERICK RD,Germantown,MD,20876,Food Service Facility,Food Service Facility License-Moderate,...,Not Applicable,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,39.17867,-77.23748
4,54291,26-2056,NEW GRAND MART FOOD COURT,GREEN PARADISE GERMANTOWN INC,13069 WISTERIA DR,Germantown,MD,20874,Food Service Facility,Food Service Facility License-High,...,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,39.17862,-77.27119
6,54301,26-2057,NEW GRAND MART,GREEN PARADISE GERMANTOWN INC,13069 WISTERIA DR,Germantown,MD,20874,Food Service Facility,Food Service Facility License-High,...,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,39.17862,-77.27119
8,54633,26-2075,Dairy Queen of Rockville,OMM FOUR LLC,2019 Veirs Mill Road,Rockville,MD,20851,Food Service Facility,Food Service Facility License-Moderate,...,Not Observed,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,In Compliance,39.07477,-77.11586


In [21]:
df_clean.isna().sum()

,0
registration_number,94
inspection_number,0
business_name,0
owner,20
address,0
city,0
state,0
zip,0
business_type,0
business_subtype,0


In [22]:
# Creating a unique key to distinguish each entity from one another and establish a group of entities.

# I was going to use the lat and longitude,  but then I noticed that there were 103 rows from the deduplicated dataset which have missing values in those colummns
# So, I have ran the following check to see what we can learn about the affected rows


missing_geo = df_clean[df_clean['latitude'].isna()]
print(f"Rows missing geo: {len(missing_geo):,}")
print(f"Unique inspection numbers affected: {missing_geo['inspection_number'].nunique():,}")
print(f"Unique business names affected: {missing_geo['business_name'].nunique():,}")

Rows missing geo: 103
Unique inspection numbers affected: 63
Unique business names affected: 42


In [32]:
# Create a unique key and count it

df_clean['facility_key_geo'] = (
    df_clean['business_name'].str.strip().str.upper() + ' | ' +
    df_clean['latitude'].astype(str) + ',' +
    df_clean['longitude'].astype(str)
)

unique_by_geo = df_clean.drop_duplicates(subset='facility_key_geo')[
    ['facility_key_geo', 'business_name', 'latitude', 'longitude']
].reset_index(drop=True)

print(f"Unique rows by geo: {len(unique_by_geo):,}")

Unique rows by geo: 4,709


In [ ]:
# Drop the rows missing long and lat



### **DELETE THIS LATER**
---

In [34]:
# Find cases where same address has inspections on the same date under different registration numbers
same_day = (
    df_clean.groupby(['address', 'inspection_start_date'])['registration_number']
    .nunique()
    .reset_index()
    .rename(columns={'registration_number': 'reg_count'})
)

# Filter to only the suspicious cases
suspicious = same_day[same_day['reg_count'] > 1].sort_values('reg_count', ascending=False)
print(f"Address/date combos with multiple registration numbers: {len(suspicious):,}")

# Pull the actual records for a closer look
sample_address = suspicious.iloc[0]['address']
sample_date = suspicious.iloc[0]['inspection_start_date']

df_clean[
    (df_clean['address'] == sample_address) &
    (df_clean['inspection_start_date'] == sample_date)
][['registration_number', 'business_name', 'address', 'inspection_start_date', 'inspection_number']].drop_duplicates()

Address/date combos with multiple registration numbers: 5,375


,registration_number,business_name,address,inspection_start_date,inspection_number
861352,15121,DC PRETZEL COMPANY,2341 DISTRIBUTION CIR,2024-11-19T00:00:00.000,24-3919
861475,15148,GOURMET & GOURMAND,2341 DISTRIBUTION CIR,2024-11-19T00:00:00.000,24-3880
861598,15605,LARMAX HOMES,2341 DISTRIBUTION CIR,2024-11-19T00:00:00.000,24-3876
861844,16143,L.I.B FLAVORS,2341 DISTRIBUTION CIR,2024-11-19T00:00:00.000,24-3920
862733,46722,DC PRETZEL COMPANY,2341 DISTRIBUTION CIR,2024-11-19T00:00:00.000,24-3919
862735,46751,GOURMET & GOURMAND,2341 DISTRIBUTION CIR,2024-11-19T00:00:00.000,24-3880
862737,48229,LARMAX HOMES,2341 DISTRIBUTION CIR,2024-11-19T00:00:00.000,24-3876
862864,53770,L.I.B FLAVORS,2341 DISTRIBUTION CIR,2024-11-19T00:00:00.000,24-3920


In [33]:
# Step 1: How often does name+address map to multiple registration numbers?
multi_reg = (
    df_clean.groupby(['business_name', 'address'])['registration_number']
    .nunique()
    .reset_index()
    .rename(columns={'registration_number': 'reg_count'})
)
print(multi_reg['reg_count'].value_counts())

# Step 2: Look at the actual cases
multi_reg[multi_reg['reg_count'] > 1].sort_values('reg_count', ascending=False).head(20)

reg_count
2    2083
3     878
1     869
4     539
5     180
6       5
0       2
7       1
Name: count, dtype: int64


,business_name,address,reg_count
748,CHICKEN POCHA,12933 WISTERIA DR.,7
3838,SUBWAY #54931,9613-F MEDICAL CENTER DR,6
1096,DOG HAUS BIERGARTEN SILVER SPRING,933 ELLSWORTH DR,6
1146,DON POLLO OF BETHESDA,7007-7009 WISCONSIN AVE.,6
1397,FAIRLAND ETHIOPIAN RESTARUANT,13310 OLD COLUMBIA PIKE,6
4130,THE WOODS ACADEMY,6801 GREENTREE RD,6
224,ANN'S CAFE,7616 AIRPARK RD,5
4301,VERONICA'S BAKERY & CAFÉ,8501 PINEY BRANCH RD.,5
3975,TACO BELL,11160 VEIRS MILL RD FC09,5
1695,GOODIE'S CARRYOUT,15420-D OLD COLUMBIA PIKE,5


----

### **END OF DELETE SECTION**

***Describe what one row represents in this dataset:***

Each row represents one inspection event at one retaurant given a specific day. Each of the columns is represents an aspect of the inspection (e.g. - Food From Approved Source (C) is an aspect of the inspection which checks if the food being used to create the food provided at the establishment is from approved sources.)






### Variable Table

| Column Name | API Field Name | Data Type | Description | Inspection Criteria | Critical Item |
|-------------|----------------|----------|-------------|---------------------|----------------|
| Registration Number | registration_number | Text | Registration number for restaurant. | No | No |
| Inspection Number | inspection_number | Text | Unique inspection identifier. | No | No |
| Business Name | business_name | Text | Name of establishment. | No | No |
| Owner | owner | Text | Name of entity owning the establishment. | No | No |
| Address | address | Text | Business address. | No | No |
| City | city | Text | City where the establishment is located. | No | No |
| State | state | Text | State where the establishment is located. | No | No |
| Zip | zip | Text | ZIP code of the establishment. | No | No |
| Business Type | business_type | Text | Classification of business. | No | No |
| Business Subtype | business_subtype | Text | Sub-classification of business type. | No | No |
| Inspection Type | inspection_type | Text | Reason for inspection (e.g., monitoring, complaint, follow-up). | No | No |
| Inspection Date | inspection_start_date | Floating Timestamp | Date of inspection. | No | No |
| Inspection Results | status | Text | Overall result of the inspection. | No | No |
| Food From Approved Source (C) | food_from_approved_source | Text | All food or food ingredients must derive from a regulated source. | Yes | Yes |
| Food Protected from Contamination (C) | food_protected_from_contamination | Text | Food must be protected from contamination and unfit food is prohibited. | Yes | Yes |
| Ill Workers Restricted (C) | workers_restricted | Text | Food handlers with transmissible diseases must not handle food. | Yes | Yes |
| Proper Hand Washing (C) | proper_hand_washing | Text | Employees must properly wash hands before handling food or food-contact surfaces. | Yes | Yes |
| Cooling Time and Temperature (C) | cooling_time_and_temperature | Text | Potentially hazardous food must be cooled within required time and temperature parameters. | Yes | Yes |
| Cold Holding Temperature (C) | cold_holding_temperature | Text | Cold food must be held at or below required temperature. | Yes | Yes |
| Hot Holding Temperature (C) | hot_holding_temperature | Text | Hot food must be held at or above required temperature. | Yes | Yes |
| Cooking Time and Temperature (C) | cooking_time_and_temperature | Text | Food must be cooked to required internal temperature for specified time. | Yes | Yes |
| Reheating Time and Temperature (C) | reheating_time_and_temperature | Text | Food must be reheated to required internal temperature within specified timeframe. | Yes | Yes |
| Hot and Cold Running Water Provided (C) | hot_and_cold_running_water_provided | Text | Establishment must provide hot and cold running water under pressure. | Yes | Yes |
| Proper Sewage Disposal (C) | proper_sewage_disposal | Text | Sewage must be properly disposed of without overflow. | Yes | Yes |
| Toxic Substances & Pesticides | toxic_substances_and_pesticides | Text | Toxic substances and pesticides must be properly labeled and stored. | Yes | No |
| Rodents and Insects | rodents_and_insects | Text | Establishment must control and prevent rodents and insects. | Yes | No |
| Nutritional Labeling | nutritional_labeling | Text | Applicable restaurant chains must display calorie information. | Yes | No |
| Trans Fat Ban | trans_fat_ban | Text | Prohibits use or storage of foods containing partially hydrogenated oils above allowed limits. | Yes | No |
| No-Smoking Sign Posted | nosmoking_sign_posted | Text | No-smoking signage must be clearly posted where required. | Yes | No |
| Location | location | Point | Geocoded location of the establishment. | No | No |

---
## 3. Data Wrangling

In [6]:
# --- Derive Per-Inspection Compliance Score ---
# Encode compliance columns as 0/1 and compute compliance rate per inspection_number


In [7]:
# --- Deduplicate to One Row Per Inspection ---
# Keep inspection-level fields (inspection_number, zip, inspection_start_date, etc.)
# and join the compliance score


In [8]:
# --- Data Cleaning ---
# Handle missing values, fix data types, filter invalid zips, etc.


*Describe any cleaning decisions made and why:*

---
## 4. Data Analyses

---
### 4a. Quantitative Variable — Descriptive Analysis

**Research Question:**

*Why is this data appropriate for answering the question?*

In [9]:
# --- Summary Statistics for Compliance Score ---


In [10]:
# --- Visualization (e.g., histogram of compliance scores) ---


*Interpret the results. Discuss:*
- *Measures of center and spread*
- *Outliers*
- *Shape of distribution (symmetric, skewed, etc.)*

---
### 4b. Categorical Variable — Descriptive Analysis

**Research Question:** *(e.g., Which zip codes have the highest number of inspections?)*

In [11]:
# --- Value Counts / Frequency Table for Zip Code ---


In [12]:
# --- Visualization (e.g., bar chart of inspections by zip code) ---


*Interpret the results. Discuss:*
- *Definition of levels (what each zip code represents)*
- *Notable patterns or concentrations*

---
### 4c. Relationship Between Variables — Exploratory Analysis

**Research Question:** *(e.g., Do compliance rates differ across zip codes?)*

*Why do you expect an association between these two variables?*

In [13]:
# --- Group compliance scores by zip code ---


In [14]:
# --- Visualization (e.g., boxplot or grouped bar chart by zip code) ---


*Interpret the results. Discuss:*
- *Whether an association exists*
- *Description of the relationship*
- *Any surprising findings*

---
### 4d. Nonparametric Inference — Median Compliance Score

**Variable:** Compliance Score  
**Method:** Bootstrap resampling (~10% random sample)

In [15]:
# --- Draw ~10% random sample ---


In [16]:
# --- Bootstrap the median ---


In [17]:
# --- Compute and display confidence interval ---


*Explain the bootstrap method used and interpret the confidence interval in context:*

---
## 5. Conclusions / Recommendations

*Write 1–2 paragraphs summarizing:*
- *Key findings from each analysis*
- *Any trends or associations observed across zip codes*
- *Suggested next steps for further analysis*